In [2]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

from langgraph.checkpoint.memory import InMemorySaver

from langgraph.types import interrupt, Command

from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel, Field

from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate

from langgraph.graph import MessagesState

from dotenv import load_dotenv
from huggingface_hub import InferenceClient
import requests, operator, random, sqlite3

from langchain_core.messages.utils import trim_messages,count_tokens_approximately

In [3]:
load_dotenv()

True

In [6]:
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    max_new_tokens=256,
    temperature=0.7
)
model = ChatHuggingFace(llm=llm)
MAX_TOKENS = 150

In [7]:
def call_mode(state: MessagesState):

    messages = trim_messages(
        state["messages"],
        strategy = "last",
        token_counter = count_tokens_approximately,
        max_tokens = MAX_TOKENS
    )
    print("Current token count", count_tokens_approximately(messages=messages))

    for message in messages:
        print(message.content)
        
    response = model.invoke(messages)
    return {"messages": [response]}

In [16]:
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_mode)
builder.add_edge(START, "call_model")
builder.add_edge("call_model", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [20]:
config = {"configurable": {"thread_id": "chat-1"}}

result = graph.invoke(
    {"messages": [{"role": "user", "content": "Hi, my name is Ammar."}]},
    config,
)

result["messages"][-1].content

Current token count 10
Hi, my name is Ammar.


"Hello Ammar! Nice to meet you. How can I assist you today? Is there anything specific you're interested in discussing or any questions you have?"

In [21]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": "I am learning LangGraph."}]},
    config,
)

result["messages"][-1].content

Current token count 62
Hi, my name is Ammar.
Hello Ammar! Nice to meet you. How can I assist you today? Is there anything specific you're interested in discussing or any questions you have?
I am learning LangGraph.


'Great! LangGraph is a powerful tool for natural language processing and graph-based analysis. It can be used for tasks like text classification, entity linking, and more. What specific aspects of LangGraph are you interested in learning about? Are you looking to understand its core concepts, how to use it for a particular task, or any specific features?'

In [22]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": "Can you explain short term memory very briefly in 1 line?"}]},
    config,
)

result["messages"][-1].content

Current token count 123
I am learning LangGraph.
Great! LangGraph is a powerful tool for natural language processing and graph-based analysis. It can be used for tasks like text classification, entity linking, and more. What specific aspects of LangGraph are you interested in learning about? Are you looking to understand its core concepts, how to use it for a particular task, or any specific features?
Can you explain short term memory very briefly in 1 line?


'Certainly! Short-term memory is a cognitive system that temporarily holds and manipulates information for a brief period, typically a few seconds to a minute.'

In [23]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config,
)

result["messages"][-1].content

Current token count 72
Can you explain short term memory very briefly in 1 line?
Certainly! Short-term memory is a cognitive system that temporarily holds and manipulates information for a brief period, typically a few seconds to a minute.
What is my name?


'Your name is not provided in our conversation context. Could you please tell me your name so I can use it appropriately?'

In [24]:
for item in graph.get_state({"configurable": {"thread_id": "chat-1"}}).values['messages']:
    print(item.content)
    print('-'*120)

Hi, my name is Ammar.
------------------------------------------------------------------------------------------------------------------------
Hello Ammar! Nice to meet you. How can I assist you today? Is there anything specific you'd like to know or discuss?
------------------------------------------------------------------------------------------------------------------------
I am learning LangGraph.
------------------------------------------------------------------------------------------------------------------------
Great to hear that you are learning LangGraph! LangGraph is a powerful framework that combines natural language processing (NLP) with knowledge graph technologies. It can be used for a variety of applications, including information retrieval, question answering, and text-to-knowledge graph mapping.

If you have any specific questions or need help with particular aspects of LangGraph, such as installation, basic usage, or advanced functionalities, feel free to ask! Here